In [1]:
# %% Imports
from jax import random, jit, numpy as jnp
from modax.data.kdv import DoubleSoliton

In [2]:
# %% Making data
key = random.PRNGKey(42)

x = jnp.linspace(-10, 10, 100)
t = jnp.linspace(0.1, 1.0, 10)
t_grid, x_grid = jnp.meshgrid(t, x, indexing="ij")

u = DoubleSoliton(x_grid, t_grid, c=[5.0, 2.0], x0=[0.0, -5.0])

X = jnp.concatenate([t_grid.reshape(-1, 1), x_grid.reshape(-1, 1)], axis=1)
y = u.reshape(-1, 1)
y += 0.2 * jnp.std(y) * random.normal(key, y.shape)

In [3]:
# %% Imports
import jax
from jax import jit, value_and_grad, numpy as jnp
from flax import linen as nn
from typing import Sequence, Tuple
from modax.feature_generators import library_backward
from modax.models.networks import MLP
from modax.losses.utils import normal_LL, precision, gamma_LL

from functools import partial
from jax import lax

from modax.training.logging import Logger
from modax.training.sparsity_scheduler import mask_scheduler

In [4]:
# Forward solver
@partial(jit, static_argnums=(0,))
def fwd_solver(f, z_init, tol=1e-4):
    def cond_fun(carry):
        z_prev, z = carry
        return (
            jnp.linalg.norm(z_prev[:-1] - z[:-1]) > tol
        )  # for numerical reasons, we check the change in alpha

    def body_fun(carry):
        _, z = carry
        return z, f(z)

    init_carry = (z_init, f(z_init))
    _, z_star = lax.while_loop(cond_fun, body_fun, init_carry)
    return z_star

In [5]:
@jit
def bayes_ridge_update_efficient(prior_params, y, X, gram, eigvals):
    # Unpacking parameters
    alpha_prev, beta_prev = prior_params

    # Calculating intermediate matrices
    n_samples, n_terms = X.shape
    gamma_ = jnp.sum((beta_prev * eigvals) / (alpha_prev + beta_prev * eigvals))
    S = jnp.linalg.inv(beta_prev * gram + alpha_prev * jnp.eye(n_terms))
    mn = beta_prev * S @ X.T @ y

    # Update estimate
    alpha = gamma_ / jnp.sum(mn ** 2)
    beta = (n_samples - gamma_) / (jnp.sum((y - X @ mn) ** 2))

    return jnp.stack([alpha, beta], axis=0)

In [18]:
@jit
def evidence(prior_params, y, X):
    alpha, beta = prior_params

    n_samples, n_terms = X.shape
    A = alpha * jnp.eye(n_terms) + beta * X.T @ X
    mn = beta * jnp.linalg.inv(A) @ X.T @ y

    E = beta * jnp.sum((y - X @ mn) ** 2) + alpha * jnp.sum(mn ** 2)
    loss = 0.5 * (
        n_terms * jnp.log(alpha)
        + n_samples * jnp.log(beta)
        - E
        - jnp.linalg.slogdet(A)[1]
        - n_samples * jnp.log(2 * jnp.pi)
    )

    # following tipping, numerically more stable if a, b -> 0 but doesn't have constant terms.
    return loss, jnp.mean((y - X @ mn) ** 2), mn

In [19]:
# Custom backprop function for iterative methods
@partial(jax.custom_vjp, nondiff_argnums=(0,))
@partial(jit, static_argnums=(0,))
def fixed_point_solver(f, args, z_init, tol=1e-5):
    z_star = fwd_solver(lambda z: f(z, *args), z_init=z_init, tol=tol)
    return z_star


@partial(jit, static_argnums=(0,))
def fixed_point_solver_fwd(f, args, z_init, tol):
    z_star = fixed_point_solver(f, args, z_init, tol)
    return z_star, (z_star, tol, args)


@partial(jit, static_argnums=(0,))
def fixed_point_solver_bwd(f, res, z_star_bar):
    z_star, tol, args = res
    _, vjp_a = jax.vjp(lambda args: f(z_star, *args), args)
    _, vjp_z = jax.vjp(lambda z: f(z, *args), z_star)
    res = vjp_a(
        fwd_solver(
            lambda u: vjp_z(u)[0] + z_star_bar, z_init=jnp.zeros_like(z_star), tol=tol
        )
    )
    return (*res, None, None)  # None for init and tol


fixed_point_solver.defvjp(fixed_point_solver_fwd, fixed_point_solver_bwd)

In [20]:
class BayesianRegression(nn.Module):
    tol: float = 1e-3

    def setup(self):
        self.update = lambda prior, y, X, gram, eigvals: bayes_ridge_update_efficient(
            prior, y, X, gram, eigvals
        )

    @nn.compact
    def __call__(self, inputs):
        is_initialized = self.has_variable("bayes", "z")
        z_init = self.variable(
            "bayes", "z", lambda y: jnp.stack([1, 1 / jnp.var(y)], axis=0), inputs[0]
        )

        # These things remain constant every iteration step, so we precalculate them
        y, X = inputs
        X_normed = X / jnp.linalg.norm(X, axis=0, keepdims=True)
        gram = X_normed.T @ X_normed
        eigvals = jnp.linalg.eigvalsh(gram)

        z_star = fixed_point_solver(
            self.update, (y, X_normed, gram, eigvals), z_init.value, tol=self.tol
        )
        if is_initialized:
            z_init.value = z_star
        return z_star

In [21]:
# Deepmodel model for bayesian regression
class Deepmod(nn.Module):
    features: Sequence[int]
    tol: float = 1e-3

    @nn.compact
    def __call__(self, inputs):
        prediction, dt, theta = library_backward(MLP(self.features), inputs)
        z = BayesianRegression(self.tol)((dt, theta))
        return prediction, dt, theta, z

In [39]:
# Loss function to use with bayes regression
def loss_fn_pinn_bayes_regression(params, state, model, x, y):
    """first argument should always be params!"""
    variables = {"params": params, **state}
    (prediction, dt, theta, z), updated_state = model.apply(
        variables, x, mutable=list(state.keys())
    )

    # MSE
    tau = 1 / jnp.mean((prediction - y))**2
    p_mse, MSE = normal_LL(prediction, y, tau)

    # Reg
    theta_normed = theta / jnp.linalg.norm(theta, axis=0, keepdims=True)
    p_reg, reg, mn = evidence(z, dt, theta_normed)
    loss = -(p_mse + p_reg)
    metrics = {
        "loss": loss,
        "p_mse": p_mse,
        "p_reg": p_reg,
        "mse": MSE,
        "reg": reg,
        "coeff": mn,
        "tau": tau,
        "beta": z[1],
        "alpha": z[0],
    }
    return loss, (updated_state, metrics, (prediction, dt, theta, mn))

In [40]:
# Function to create fast update for flax models with variables
def create_update(loss_fn, *args, **kwargs):
    def step(opt, state, loss_fn, *args, **kwargs):
        grad_fn = value_and_grad(loss_fn, argnums=0, has_aux=True)
        (loss, (updated_state, metrics, output)), grad = grad_fn(
            opt.target, state, *args, **kwargs
        )
        opt = opt.apply_gradient(grad)  # Return the updated optimizer with parameters.

        return (opt, updated_state), metrics, output

    return jit(lambda opt, state: step(opt, state, loss_fn, *args, **kwargs))

In [41]:
from sklearn.model_selection import train_test_split

In [50]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [51]:
from flax import optim

In [52]:
# %% Building model and params
model = Deepmod([30, 30, 30, 1])
variables = model.init(key, X)

optimizer = optim.Adam(learning_rate=2e-3, beta1=0.99, beta2=0.99)
state, params = variables.pop("params")
optimizer = optimizer.create(params)

In [53]:
update = create_update(loss_fn_pinn_bayes_regression, model=model, x=X_train, y=y_train)
validation_metric = jit(
    lambda opt, state: loss_fn_pinn_bayes_regression(opt.target, state, model, X_test, y_test)[1][1]
)
logger = Logger()
scheduler = mask_scheduler(delta=0.0, patience=1000)

In [54]:
  (optimizer, state), train_metrics, output = update(optimizer, state)

In [55]:
output[3]

DeviceArray([[ 0.22210482],
             [ 1.0659145 ],
             [ 1.7774267 ],
             [ 0.30221984],
             [ 2.7213964 ],
             [-2.802231  ],
             [-2.4625847 ],
             [-0.28356966],
             [-1.4723375 ],
             [ 2.1919117 ],
             [ 1.2147497 ],
             [ 0.15689623]], dtype=float32)

In [56]:
# Training till mse
max_epochs = 1e4
for epoch in jnp.arange(max_epochs):
    (optimizer, state), train_metrics, output = update(optimizer, state)
    prediction, dt, theta, coeffs = output

    if epoch % 10 == 0:
        print(f"Loss step {epoch}: {train_metrics['loss']}")

logger.close()


Loss step 0.0: -809.7049560546875
Loss step 10.0: -938.8602294921875
Loss step 20.0: -1598.222900390625
Loss step 30.0: -1750.9036865234375
Loss step 40.0: -2276.79150390625
Loss step 50.0: -2454.258544921875
Loss step 60.0: -2714.52197265625
Loss step 70.0: -3119.7724609375
Loss step 80.0: -3189.562744140625
Loss step 90.0: nan
Loss step 100.0: nan
Loss step 110.0: nan
Loss step 120.0: nan
Loss step 130.0: nan
Loss step 140.0: nan
Loss step 150.0: nan
Loss step 160.0: nan
Loss step 170.0: nan
Loss step 180.0: nan
Loss step 190.0: nan
Loss step 200.0: nan
Loss step 210.0: nan
Loss step 220.0: nan
Loss step 230.0: nan
Loss step 240.0: nan
Loss step 250.0: nan
Loss step 260.0: nan
Loss step 270.0: nan
Loss step 280.0: nan
Loss step 290.0: nan
Loss step 300.0: nan
Loss step 310.0: nan
Loss step 320.0: nan
Loss step 330.0: nan
Loss step 340.0: nan
Loss step 350.0: nan
Loss step 360.0: nan
Loss step 370.0: nan
Loss step 380.0: nan
Loss step 390.0: nan
Loss step 400.0: nan
Loss step 410.0: n

KeyboardInterrupt: 